In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%config InlineBackend.figure_format='svg'


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


from mpl_toolkits.mplot3d import Axes3D

from pydmd import DMD
from pydmd.bopdmd import BOPDMD
from pydmd.plotter import plot_eigs, plot_summary
from pydmd.preprocessing.hankel import hankel_preprocessing

from morphing_birds import Hawk3D, plot_plotly, animate_plotly, animate_plotly_compare, animate, animate_compare

# from BirdDMD import plot_markers_overDist, plot_2d_markers, plot_single_sequence, plot_score_multi_PCs, reorder_dmd_results, reconstruct_dmd

from BirdDMD import (load_sequence_data, 
                     remove_time_duplicates,
                     run_single_wingbeat_dmd,
                     reconstruct_dmd,
                     run_forecast,
                     reorder_dmd_results,
                     plot_amplitude_ranking,
                     plot_2d_markers, 
                     plot_markers_overDist, 
                     plot_single_sequence, 
                     plot_score_multi_PCs)

np.set_printoptions(suppress=True, precision=3)
# Make matplotlib use the font Andale Mono
plt.rcParams['font.family'] = 'Andale Mono'




In [ ]:
column_names = np.load("../data/samples/ColumnNames2.npz")
file = np.load("../data/samples/Initial_9mStraightTurnToothless_Bilateral2.npz", allow_pickle=True)
marker_data = file["marker_data"]
info_data = file["info_data"]

marker_column_names = column_names["marker_column_names"]
info_column_names = column_names["info_column_names"]

# Create a pandas dataframe
marker_df = pd.DataFrame(marker_data, columns=marker_column_names)
info_df = pd.DataFrame(info_data, columns=info_column_names)

# Concatenate the dataframes vertically
wingbeat_df = pd.concat([info_df,marker_df], axis=1)

# Number of sequences
n_sequences = wingbeat_df["seqID"].nunique()
print("Number of sequences: ", n_sequences)

print(str(wingbeat_df.shape) + " dataframe loaded." )
wingbeat_df.head()

## Average Shape of the Hawk

Within the data set there are 8 markers, which are used to represent the hawk's wing and tail. All markers are represented as relative to the centre of mass. 

The markers are named as follows:

- Left Wingtip   (similar to the fingertip)
- Right Wingtip  (similar to the fingertip)
- Left Primary   (similar to the wrist)
- Right Primary  (similar to the wrist)
- Left Secondary (trailing edge of the wing)
- Right Secondary (trailing edge of the wing)
- Left Tailtip
- Right Tailtip

You can see the average shape of the hawk by plotting the markers in 3D space. The blue points are measured by the motion capture, the rest of the grey points are just for visualisation purposes and are estimated using measurements of body size in the hawks.

In [ ]:
hawk3d = Hawk3D("../data/mean_hawk_shape.csv")

plot_plotly(hawk3d)

## Plot Raw Data

This is a single initial wingbeat from take-off by one hawk called Toothless. There are around 175 flights at 250 fps. We can plot them over the flight to see the markers changing in x, y, and z. 

Note the x axis is the horizontal distance to the perch as measured at the centre of mass of the bird. 

In [ ]:
plot_markers_overDist(wingbeat_df, marker_column_names, x_axis='time')

# Plot the Marker Trajectories in 2d

In [ ]:
plot_2d_markers(wingbeat_df, marker_column_names)

## Find Sequence Lengths

In [ ]:
# Using seqID as a category, find the number of rows in each category
seqID_counts = wingbeat_df['seqID'].value_counts()

# Create a histogram
plt.figure(figsize=(3,3))
plt.hist(seqID_counts, bins=20)
plt.xlabel("Number of frames per sequence")

plt.show()

# Find the sequences with the most frames
seqID_counts.idxmax()

# Find what number 04_09_048_1 is in the list of unique sequences
# wingbeat_df['seqID'].unique().tolist().index('04_09_048_1')
wingbeat_df['seqID'].unique().tolist().index('04_09_051_2')


## Plot a Single Sequence

In [ ]:
plt.close("all")
plot_single_sequence(wingbeat_df, 60, marker_name="right_wingtip_z", x_axis="time")


In [ ]:
hawk3d = Hawk3D("../data/mean_hawk_shape.csv")
markers, times = load_sequence_data(wingbeat_df, '04_09_048_1', marker_column_names)

markers = markers.reshape(-1, 8, 3)

animate_plotly_compare(
        hawk3d,
        keypoints_frames_list=[
            markers,             # Use the keypoints returned by the function
        ])

## Remove Duplicated Time Frames

In [ ]:
wingbeat_df = remove_time_duplicates(wingbeat_df)

## Test how many Modes are appropriate

In [ ]:

fig, ax, sorted_amps = plot_amplitude_ranking(
        markers,
        times,
        max_modes=20,
        d=2, # Match delay used in main analysis
        eig_constraints={"conjugate_pairs"} # Match notebook constraint
    )
plt.show() # Display the plot generated by the function


In [ ]:
print(dmd_results.modes[0,:])

## DMD on Single Wingbeat Example

In [ ]:

times, markers, Lambda, Modes, bn, Psi, phase_shifts, dmd_results, keypoints = run_single_wingbeat_dmd(
    bird_name ="Toothless",
    perch_dist="12m",
    turn="Straight",
    behaviour="Initial",
    n_modes=8,
    d=2,
    verbose=True
)



## Compare Original Sequence with DMD Reconstruction

In [ ]:
hawk3d = Hawk3D("../data/mean_hawk_shape.csv")

print("Original Markers Shape:", markers.shape)
print("DMD Keypoints Shape:", keypoints.shape)

# Take away the first frame and add an extra frame to the end
keypoints = np.concatenate([keypoints[1:], keypoints[-1:]], axis=0)

animate_plotly_compare(
        hawk3d,
        keypoints_frames_list=[
            keypoints, # Use the reshaped original markers
            markers             # Use the keypoints returned by the function
        ])


## Upsample the Data



In [ ]:
# Original time points
# print(times)
print(times.shape)
# Create interpolated time points for DMD reconstruction
fake_time = np.arange(times[0], times[-1]+0.001, 0.001)
# print(fake_time)

# Create expanded original data by repeating frames
expanded_markers = []
current_frame = 0

for t in fake_time:
    # Keep using same frame until we hit a time point in the original data
    while current_frame < len(times)-1 and t >= times[current_frame+1]:
        current_frame += 1
    expanded_markers.append(markers[current_frame].copy())

expanded_markers = np.array(expanded_markers)
expanded_markers = expanded_markers.reshape(-1, 8, 3)

print(expanded_markers.shape)

hawk3d = Hawk3D("../data/mean_hawk_shape.csv")
average_shape = hawk3d.markers
num_markers = 8
keypoints = run_forecast(dmd_results, fake_time, average_shape, num_markers)

# Animate both sequences
animate_plotly_compare(hawk3d, keypoints_frames_list=[expanded_markers, keypoints], colours=["blue", "red"])

## Find Mode Frequencies

In [ ]:
np.imag(dmd_results.eigs)

In [ ]:

n_Modes = 8
Lambda, Modes, bn, Psi, PhaseShifts = reorder_dmd_results(dmd_results, num_markers=8, nModes=n_Modes)
for i_Mode in range(n_Modes):
    print(f"Frequency of Mode {i_Mode}: {Lambda[i_Mode]}")

In [ ]:
plt.plot(times, keypoints[:, 0, 2], label='Marker 1 Z')
plt.plot(times, keypoints[:, 1, 2], label='Marker 2 Z')
plt.xlabel("Time (s)")
plt.ylabel("Z Position")
plt.title("Forecasted DMD Z Trajectories")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

# 📊 Plot example marker trace
plt.figure(figsize=(5, 3))
plt.plot(times, markers[:, 0], label="Marker 1 (x)")
plt.plot(times, keypoints[:, 0, 0], label="DMD forecast (x)", linestyle="--")
plt.xlabel("Time (s)")
plt.ylabel("Position (m)")
plt.title("Marker vs Forecast (DMD)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
original_markers = markers.reshape(-1, 8, 3)
hawk3d = Hawk3D("../data/mean_hawk_shape.csv")
animate_plotly_compare(hawk3d, keypoints_frames_list=[original_markers, keypoints], colours=["blue", "red"])